In [23]:
import sys
sys.path.append('../')
from swiss_code.excel import excel
from tests import dataset_synth
import xlwings as xw

%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [24]:
wb = excel.get_or_create_workbook("test.xlsx")

In [26]:
df = dataset_synth.simulate_df(1000)
df.head()

,transaction_id,customer_id,amount,payment_method,category,date
0,1,1060,310.48,Cash,Electronics,2024-01-01
1,2,1552,448.52,Cash,Entertainment,2024-01-02
2,3,1404,107.01,Credit Card,Clothing,2024-01-03
3,4,2461,497.05,Credit Card,Electronics,2024-01-04
4,5,4044,443.41,Cash,Entertainment,2024-01-05


In [27]:
output = df.pivot_table(
    index="payment_method", columns="category", values="amount", aggfunc="sum"
)
output

category,Clothing,Electronics,Entertainment,Groceries
payment_method,,,,
Cash,13638.09,20550.89,13295.60,16148.28
Credit Card,18399.45,15196.52,13953.11,17013.66
Debit Card,14378.39,13630.95,13443.22,17043.61
PayPal,14815.06,14672.94,15558.88,17882.21


In [28]:
first_sheet = excel.select_sheet('first_test', wb)
excel.write_df_to_excel(output, first_sheet)

In [29]:
transact_by_month = (
    df.assign(
        month=df["date"].dt.month.astype(str) + "-" + df["date"].dt.year.astype(str)
    )
    .pivot_table(
        index=["month", "payment_method"],
        columns="category",
        values="amount",
        aggfunc="sum",
    )
    .fillna(0)
)
transact_by_month

category               Clothing  Electronics  Entertainment  Groceries
month  payment_method                                                 
1-2024 Cash              330.09       343.84        1211.62     440.26
       Credit Card       573.13      1090.44         605.89     518.33
       Debit Card          0.00       321.51         754.51     387.70
       PayPal            460.57        66.98         197.78     494.00
1-2025 Cash              511.78      1208.98           0.00       0.00
...                         ...          ...            ...        ...
9-2025 PayPal            458.83       434.22         801.61     911.79
9-2026 Cash              507.12       828.20         552.93     495.04
       Credit Card         0.00       104.16         739.32     360.42
       Debit Card        467.29       389.04         194.57     634.25
       PayPal            727.12        85.43           0.00       0.00

[132 rows x 4 columns]

In [30]:
second_sheet = excel.select_sheet('second_sheet', wb)
excel.write_df_to_excel(transact_by_month, second_sheet)
excel.merge_column(second_sheet, col=1)

In [31]:
excel.close_out_book(wb)